In [10]:
# importing basic libraries

import pandas as pd
import numpy as np

In [11]:
# importing training data

training_data=pd.read_csv('cleaned_training_data.csv')

In [12]:
# splitting our training data into independent and dependent variables

X=training_data.drop(columns=['Diagnosis'], axis=1)
y=training_data['Diagnosis']

In [13]:
# splitting our data into train and test splits

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=12)

In [14]:
# creating a basic logistic regression model

from sklearn.linear_model import LogisticRegression
logreg_basic=LogisticRegression()
logreg_basic.fit(X_train, y_train)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [15]:
# making predictions

y_pred=logreg_basic.predict(X_test)

In [16]:
# evaluating our model

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\nClassification Report:\n{classification_report(y_test, y_pred)}")

Confusion Matrix:
[[45  0]
 [ 4 31]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        45
           1       1.00      0.89      0.94        35

    accuracy                           0.95        80
   macro avg       0.96      0.94      0.95        80
weighted avg       0.95      0.95      0.95        80



In [20]:
# Implementing GridSearchCV on our Model

params={
    'penalty':['l1', 'l2', 'elasticnet', None],
    'solver':['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    'max_iter':[10,50,100,200,500,1000],
    'multi_class':['auto', 'ovr', 'multinomial']

}

from sklearn.model_selection import GridSearchCV, StratifiedKFold
cv=StratifiedKFold(n_splits=5)
grid=GridSearchCV( estimator=logreg_basic, param_grid=params, cv=cv, n_jobs=-1, scoring='accuracy')
grid.fit(X_train, y_train)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logis

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=LogisticRegression(), n_jobs=-1,
             param_grid={'max_iter': [10, 50, 100, 200, 500, 1000],
                         'multi_class': ['auto', 'ovr', 'multinomial'],
                         'penalty': ['l1', 'l2', 'elasticnet', None],
                         'solver': ['lbfgs', 'liblinear', 'newton-cg',
                                    'newton-cholesky', 'sag', 'saga']},
             scoring='accuracy')

In [21]:
grid.best_score_

np.float64(0.9811507936507937)

In [22]:
grid.best_params_

{'max_iter': 50,
 'multi_class': 'multinomial',
 'penalty': 'l2',
 'solver': 'lbfgs'}

In [26]:
# training our best model

logreg_bestmodel=grid.best_estimator_
logreg_bestmodel.fit(X_train, y_train)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1237: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=50, multi_class='multinomial')

In [27]:
# evaluating our best model

y_pred_best=logreg_bestmodel.predict(X_test)

print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred_best)}\nClassification Report:\n{classification_report(y_test, y_pred_best)}")

Confusion Matrix:
[[45  0]
 [ 4 31]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        45
           1       1.00      0.89      0.94        35

    accuracy                           0.95        80
   macro avg       0.96      0.94      0.95        80
weighted avg       0.95      0.95      0.95        80



In [29]:
# exporting our best model

import pickle
with open("logreg.pkl","wb") as f:
    pickle.dump(obj=logreg_bestmodel, file=f)
